# M05-02 — Acumulados

Referencia de validación. El alumno trabaja en `notebooks/alumno/M05-02-acumulados.ipynb`.


## Celda 0 — localizar el repo


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
from pyspark.sql.functions import col, min as fmin, sum as fsum, row_number
from pyspark.sql.window import Window
spark = get_spark("novashop-m05")
sales = (
    spark.read.parquet(str(STAGING / "fact_lines"))
    .join(spark.read.parquet(str(STAGING / "customers_clean")), "customer_id", "inner")
    .where(col("is_billable"))
)
orders_gmv = sales.groupBy("customer_id", "order_id").agg(
    fmin("order_ts").alias("order_ts"), fsum("gmv_line").alias("gmv")
)
print(orders_gmv.count())
assert orders_gmv.count() == 469
w = Window.partitionBy("customer_id").orderBy("order_ts")
hist = orders_gmv.withColumn("order_n", row_number().over(w)).withColumn("gmv_running", fsum("gmv").over(w))
hist.orderBy("customer_id", "order_n").show(12)
hist.groupBy((col("order_n") == 1).alias("is_first")).count().show()
assert hist.where(col("order_n") == 1).count() == 211
print("M05-02 OK")
